In [20]:
import pandas as pd

df = pd.read_csv("D:\DHV301\salary_survey_raw.csv")
print(f'Shape: {df.shape}')

Shape: (2800, 17)


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   timestamp                        2800 non-null   object
 1   how_old_are_you                  2800 non-null   object
 2   industry                         2800 non-null   object
 3   job_title                        2800 non-null   object
 4   additional_context_on_job_title  1831 non-null   object
 5   annual_salary                    2409 non-null   object
 6   additional_monetary_comp         1262 non-null   object
 7   currency                         2800 non-null   object
 8   income_context                   531 non-null    object
 9   country                          2685 non-null   object
 10  us_state                         987 non-null    object
 11  city                             1238 non-null   object
 12  years_of_experience_in_field     2

--Loại 1: NULL HỢP LỆ - free-text filed --------------------
- LÝ DO: cột này là optional - người dùng có thể bỏ qua
- => Không cần xử lý. Ghi chú trong Report.
- df['additional_context'] -> giữ nguyên

--Loại 2: NULL CẦN ĐIỀN - biến số phân phối lớn ------------
- Hỏi lớp: 'Age phân phối như thế nào? xem histogram trước'

df['how_old_are_you'].value_counts()
- Thấy phân phối -> quyết định median hay mean

- LÝ DO: age thường phân phối lệch phải ở dataset lao động
- Median robust hơn mean khi có outliear (Người rất già/rất trẻ)

age_median = df['how_old_are_you'].mode()[0] # biến ordinal

df['how_old_are_you'] = df['how_old_are_you'].fillna(age_median)

--Loại 3: NULL CẦN ĐIỀU TRA - biến quan trọng --------------
- LÝ DO: salary là biến mục tiêu chính - không thể tùy tiện điền
- Chiến lược: giữ null, flag để theo dõi
  
df['salary_was_null'] = df['annual_salary_USD'].isnull.astype(int)
- => Điền sau bằng median theo nhóm (industry + experience)
  
industry_median = df.groupby('industry')['annual_salary_USD'].transform('median')

df['annual_salary_USD'] = df['annual_salary_USD'].fillna(industry_median)

--Kiểm tra kết quả ------------------------------------------

print('Null còn lại:')

print(df.isnull().sum()[df.isnull().sum() > 0])

1. **Tại sao chúng ta không fillna(mean) cho cột lương?**
  - Vì lương thường lệch mạnh, có ngoại lệ lớn, nên mean dễ bị kéo lên hoặc kéo xuống và làm méo phân phối dữ liệu.
  - Mean không phù hợp vì: mean không đại diện tốt cho “mức lương điển hình”, lương thường không phân phối đều, nếu có nhiều vị trí cấp cao thì sẽ có vài mức lương rất cao có thể làm trung bình tăng mạnh, thay vào ô thiếu có thể tạo ra giá trị không thực tế so với nhóm nhân viên cùng và khác cấp/bộ phận.

2. Xữ lý cột 'country' (null = 8%)

Với tỷ lệ missing value (null) nhỏ (8%) đối với cột 'country', cũng với việc đây là cột không quá quan trọng cho việc phân tích, có thể xử lý như sau:
- Hoàn toàn drop cột 'country' khỏi dataset: df.dropna('country')
- Gán giá trị 'Unknown' vào cho missing values: df['country'].fillna('Unknown')

3. Xử lý cột 'years_of_experience' (null = 3%) , có cách nào làm mất ít thông tin hơn?
   - Gán giá trị theo phân vị (Median / Mode / Mean Imputation)
   - Gán giá trị có điều kiện (Conditional Imputation)
   - Dự đoán bằng thuật toán Machine Learning (K-NN Imputation)
   - Coi "Null" là một thông tin có ý nghĩa (Category Indicator)
   - Dự báo hồi quy (Regression Imputation)

In [22]:
# 4. Viết hàm missing_report() ngắn gọn
def missing_report(df):
    miss = df.isnull().sum()
    pct = (miss / len(df) * 100).round(2)
    return (pd.DataFrame({'count': miss, 'pct': pct})
              .query('count > 0').sort_values('pct', ascending=False))

print(missing_report(df))

                                 count    pct
income_context                    2269  81.04
us_state                          1813  64.75
city                              1562  55.79
additional_monetary_comp          1538  54.93
additional_context_on_job_title    969  34.61
race                               766  27.36
annual_salary                      391  13.96
country                            115   4.11
gender                              69   2.46


In [23]:
# Số lượng Duplicated trong dataset
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup} ({n_dup/len(df)*100:.1f}%)')

Duplicate rows: 38 (1.4%)


In [24]:
# Xử lý Duplicated trong dataset
df.drop_duplicates(inplace=True)

n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup} ({n_dup/len(df)*100:.1f}%)')

Duplicate rows: 0 (0.0%)


In [25]:
# Xử lý missing values
# Drop cột missing quá cao
df = df.drop(columns=['income_context'])

# Drop rows cho cột quan trọng
df = df.dropna(subset=['annual_salary'])

#fillna cho categorical/text
df['us_state'] = df['us_state'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['race'] = df['race'].fillna('Prefer not to say')
df['gender'] = df['gender'].fillna('Prefer not to say')
df['additional_context_on_job_title'] = df['additional_context_on_job_title'].fillna('none')

#fillna
df['additional_monetary_comp'] = df['additional_monetary_comp'].fillna(0)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2374 entries, 0 to 2799
Data columns (total 16 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   timestamp                        2374 non-null   object
 1   how_old_are_you                  2374 non-null   object
 2   industry                         2374 non-null   object
 3   job_title                        2374 non-null   object
 4   additional_context_on_job_title  2374 non-null   object
 5   annual_salary                    2374 non-null   object
 6   additional_monetary_comp         2374 non-null   object
 7   currency                         2374 non-null   object
 8   country                          2374 non-null   object
 9   us_state                         2374 non-null   object
 10  city                             2374 non-null   object
 11  years_of_experience_in_field     2374 non-null   object
 12  years_of_experience_overall      2374 n

In [27]:
# dtypes của dataset trước khi xử lý
print(df.dtypes)

timestamp                          object
how_old_are_you                    object
industry                           object
job_title                          object
additional_context_on_job_title    object
annual_salary                      object
additional_monetary_comp           object
currency                           object
country                            object
us_state                           object
city                               object
years_of_experience_in_field       object
years_of_experience_overall        object
highest_level_of_education         object
gender                             object
race                               object
dtype: object


In [28]:
# dtypes của dataset sau khi xử lý
# Cột lương: object => float (xóa dấu phẩy và ký hiệu $)
for col in ['annual_salary', 'additional_monetary_comp']:
    if col in df.columns:
        # Xóa ký tự không cần thiết
        df[col] = df[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)
        # Chuyển sang numeric
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Cột ngày: object => datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Cột category: object => category (tiết kiệm bộ nhớ)
df['industry'] = df['industry'].astype('category')
df['job_title'] = df['job_title'].astype('category')
df['currency'] = df['currency'].astype('category')
df['country'] = df['country'].astype('category')
df['us_state'] = df['us_state'].astype('category')
df['city'] = df['city'].astype('category')
df['highest_level_of_education'] = df['highest_level_of_education'].astype('category')
df['gender'] = df['gender'].astype('category')
df['race'] = df['race'].astype('category')

# Kiểm tra sau khi convert
df.dtypes # phải thấy float64, datetime64, category

timestamp                          datetime64[ns]
how_old_are_you                            object
industry                                 category
job_title                                category
additional_context_on_job_title            object
annual_salary                             float64
additional_monetary_comp                  float64
currency                                 category
country                                  category
us_state                                 category
city                                     category
years_of_experience_in_field               object
years_of_experience_overall                object
highest_level_of_education               category
gender                                   category
race                                     category
dtype: object